# Surya OCR Test — ID Card OCR

Surya OCR: 0.97 accuracy, supports Arabic + French, CPU-friendly.

Based on modified Donut model with GQA and MoE layers.

**Updated for surya-ocr v0.17.x API**

In [ ]:
# Step 1: Install Surya OCR
!pip install surya-ocr

print("\nInstallation complete!")

In [ ]:
# Step 2: Verify installation (v0.17.x API)
import surya
print(f"Surya OCR version: {surya.__version__ if hasattr(surya, '__version__') else 'installed'}")

from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
print("All modules imported OK")

In [ ]:
# Step 3: Load models (downloads on first run ~500MB)
import time

print("Loading foundation model...")
t0 = time.time()
foundation_predictor = FoundationPredictor()
print(f"Foundation model loaded ({time.time()-t0:.1f}s)")

print("Loading detection model...")
t0 = time.time()
det_predictor = DetectionPredictor()
print(f"Detection model loaded ({time.time()-t0:.1f}s)")

print("Loading recognition model...")
t0 = time.time()
rec_predictor = RecognitionPredictor(foundation_predictor)
print(f"Recognition model loaded ({time.time()-t0:.1f}s)")

print("\nModels ready!")

In [ ]:
# Step 4: Upload image (for Colab)
try:
    from google.colab import files
    print("Upload your card image:")
    uploaded = files.upload()
    IMAGE_PATH = list(uploaded.keys())[0]
except:
    # Local: use existing file
    IMAGE_PATH = "front_cropped.jpg"

print(f"Using: {IMAGE_PATH}")

In [ ]:
# Step 5: Load and display image
from PIL import Image
import matplotlib.pyplot as plt

image = Image.open(IMAGE_PATH)
print(f"Image size: {image.size[0]}x{image.size[1]}")

plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.title(IMAGE_PATH)
plt.axis('off')
plt.show()

In [ ]:
# Step 6: Run OCR on full image (Arabic + French)
print("Running Surya OCR (Arabic + French)...")
print("This may take 30-60 seconds on CPU...")

t0 = time.time()

# Run recognition with detection
predictions = rec_predictor([image], det_predictor=det_predictor)

elapsed = time.time() - t0
print(f"\nOCR completed in {elapsed:.1f}s")

In [ ]:
# Step 7: Display results
print("=" * 60)
print("OCR RESULTS")
print("=" * 60)

# Check result structure and display
for page_pred in predictions:
    # v0.17.x uses text_lines attribute
    if hasattr(page_pred, 'text_lines'):
        for line in page_pred.text_lines:
            text = line.text if hasattr(line, 'text') else str(line)
            conf = line.confidence if hasattr(line, 'confidence') else 0.0
            print(f"[{conf:.2f}] {text}")
    else:
        # Fallback: print entire prediction
        print(page_pred)

print("\n" + "=" * 60)

In [ ]:
# Step 8: Visualize detections
import cv2
import numpy as np

img_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

for page_pred in predictions:
    if hasattr(page_pred, 'text_lines'):
        for line in page_pred.text_lines:
            if hasattr(line, 'bbox'):
                bbox = line.bbox
                x1, y1, x2, y2 = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
                cv2.rectangle(img_cv, (x1, y1), (x2, y2), (0, 255, 0), 2)
            elif hasattr(line, 'polygon'):
                # Some versions use polygon instead of bbox
                pts = np.array(line.polygon, dtype=np.int32)
                cv2.polylines(img_cv, [pts], True, (0, 255, 0), 2)

plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
plt.title("Surya OCR Detections")
plt.axis('off')
plt.show()

---
## Field Crop Testing

Test OCR on specific field regions.

In [ ]:
PADDING = 10

# Field definitions for 856x540 normalized card
FRONT_FIELDS = {
    "first_name_fr":      {"x": 0,   "y": 148, "w": 500, "h": 52,  "lang": ["fr"]},
    "last_name_fr":       {"x": 0,   "y": 218, "w": 500, "h": 50,  "lang": ["fr"]},
    "date_of_birth":      {"x": 160, "y": 255, "w": 210, "h": 54,  "lang": ["en"]},
    "place_of_birth_fr":  {"x": 0,   "y": 322, "w": 500, "h": 50,  "lang": ["fr"]},
    "expiry_date":        {"x": 200, "y": 365, "w": 200, "h": 45,  "lang": ["en"]},
    "first_name_ar":      {"x": 330, "y": 130, "w": 270, "h": 48,  "lang": ["ar"]},
    "last_name_ar":       {"x": 330, "y": 204, "w": 270, "h": 48,  "lang": ["ar"]},
    "card_number":        {"x": 590, "y": 405, "w": 220, "h": 42,  "lang": ["en"]},
    "gender":             {"x": 800, "y": 400, "w": 56,  "h": 45,  "lang": ["en"]},
}

print(f"Defined {len(FRONT_FIELDS)} fields")

In [ ]:
def test_field_surya(img, x, y, w, h, field_name, langs, padding=PADDING):
    """
    Crop a field and run Surya OCR on it.
    """
    iw, ih = img.size
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(iw, x + w + padding)
    y2 = min(ih, y + h + padding)

    crop = img.crop((x1, y1, x2, y2))

    # Run OCR with v0.17.x API
    result = rec_predictor([crop], det_predictor=det_predictor)

    # Extract text
    texts = []
    confs = []
    for page_pred in result:
        if hasattr(page_pred, 'text_lines'):
            for line in page_pred.text_lines:
                texts.append(line.text if hasattr(line, 'text') else str(line))
                confs.append(line.confidence if hasattr(line, 'confidence') else 0.5)

    text = " ".join(texts) if texts else ""
    conf = min(confs) if confs else 0.0

    # Display
    plt.figure(figsize=(10, 1.5))
    plt.imshow(crop)
    plt.title(f"{field_name}: '{text}' (conf: {conf:.2f})")
    plt.axis('off')
    plt.show()

    return {"text": text, "conf": conf}

print("Field test function defined.")

In [ ]:
# Test all fields
print(f"Testing {len(FRONT_FIELDS)} fields with Surya OCR")
print("This will take a few minutes on CPU...")
print("=" * 60)

all_results = {}

for field_name, field in FRONT_FIELDS.items():
    print(f"\nProcessing: {field_name}")
    result = test_field_surya(
        image,
        field["x"], field["y"], field["w"], field["h"],
        field_name,
        langs=field["lang"]
    )
    all_results[field_name] = result

# Summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for name, r in all_results.items():
    status = ">>" if r["conf"] > 0.7 else "!!"
    print(f"  {status} {name:20s} -> '{r['text']}' (conf: {r['conf']:.2f})")

---
## MRZ Reading (Back Side)

For MRZ, we can also use `readmrz` for better accuracy.

In [ ]:
# Install readmrz for MRZ
!pip install readmrz -q

print("readmrz installed")

In [ ]:
# Upload back side image
try:
    from google.colab import files
    print("Upload back side image:")
    uploaded = files.upload()
    BACK_PATH = list(uploaded.keys())[0]
except:
    BACK_PATH = "back.jpeg"

print(f"Using: {BACK_PATH}")

In [ ]:
# Read MRZ with readmrz
from readmrz import MrzDetector, MrzReader

detector = MrzDetector()
reader = MrzReader()

try:
    # Load and detect MRZ
    mrz_image = detector.read(BACK_PATH)
    cropped = detector.crop_area(mrz_image)
    
    # Display cropped MRZ
    plt.figure(figsize=(14, 3))
    plt.imshow(cropped, cmap='gray')
    plt.title("Detected MRZ Zone")
    plt.axis('off')
    plt.show()
    
    # Read MRZ
    result = reader.process(cropped)
    
    print("\nMRZ Parsed Data:")
    print("=" * 60)
    for key, value in result.items():
        print(f"  {key:20s}: {value}")
        
except Exception as e:
    print(f"MRZ reading failed: {e}")
    print("Try with Surya OCR instead...")

In [ ]:
# Alternative: Read MRZ with Surya OCR
print("Reading MRZ with Surya OCR...")

back_img = Image.open(BACK_PATH)

# Rotate if needed
if back_img.size[1] > back_img.size[0]:
    back_img = back_img.rotate(90, expand=True)

# Crop MRZ zone (bottom 28%)
mrz_y = int(back_img.size[1] * 0.72)
mrz_crop = back_img.crop((0, mrz_y, back_img.size[0], back_img.size[1]))

plt.figure(figsize=(14, 3))
plt.imshow(mrz_crop)
plt.title("MRZ Zone")
plt.axis('off')
plt.show()

# OCR with v0.17.x API
mrz_result = rec_predictor([mrz_crop], det_predictor=det_predictor)

print("\nMRZ Lines:")
for page_pred in mrz_result:
    if hasattr(page_pred, 'text_lines'):
        for line in page_pred.text_lines:
            text = line.text if hasattr(line, 'text') else str(line)
            conf = line.confidence if hasattr(line, 'confidence') else 0.0
            print(f"  [{conf:.2f}] {text}")

---
## Compare: Surya vs Previous Results

Fill in after running all notebooks:

| Field | Tesseract | EasyOCR | Surya | Winner |
|-------|-----------|---------|-------|--------|
| first_name_fr | ? | ? | ? | ? |
| last_name_fr | ? | ? | ? | ? |
| date_of_birth | ? | ? | ? | ? |
| first_name_ar | ? | ? | ? | ? |
| last_name_ar | ? | ? | ? | ? |
| card_number | ? | ? | ? | ? |